In [1]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [2]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from flatten_json import flatten
#warnings.filterwarnings('ignore')

In [3]:
# Time
start = dt.datetime(2019,4,30)
end = dt.datetime(2019,5,2)
end2 = dt.datetime(2019,5,14)
print(start,end,end-start)

2019-04-30 00:00:00 2019-05-02 00:00:00 2 days, 0:00:00


In [6]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname'])
c_users = cursor.superstars.users
aw = []
for documents in c_users.find({'created_at': {'$lt': end, '$gte': start}},{"sign_up_details":1, "created_at":1,"login_details":1}):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
df = pd.DataFrame(dic_flattened)
df.head()
df = df[df["sign_up_details_app_platform"] == "UNITY_Android"]
#df = df[df["sign_up_details_device_id"].isin(devices)]
users = df[["_id","created_at","sign_up_details_device_id","login_details_last_request_at"]]
users.columns = ["user_id","createtime","device_id","last_request"]
len(users)
users.head()

,user_id,createtime,device_id,last_request
0,5cc79098304ded1c8d247a15,2019-04-30 00:02:32.274,5dc87aa215fd9b54862319b4eedcb47c,2019-04-30 00:28:40.091
1,5cc7986cc757b72b3a1c8681,2019-04-30 00:35:56.652,471d3d6ca4b508670091bc37d6aa8f5b,2019-04-30 00:59:47.981
2,5cc798b6c1811e2b616907ef,2019-04-30 00:37:10.226,0f3d94b651bfdd544329b262c988425b,2019-04-30 00:37:36.007
3,5cc79982aaefa53d8f9ac785,2019-04-30 00:40:34.006,eee66137f6f9087db61fb80e3838a931,2019-04-30 07:50:23.633
4,5cc79b24c757b72b3a1c89ec,2019-04-30 00:47:32.593,c6bed96a0fc1a5b9b1292d3473b2a529,2019-04-30 07:54:20.873


In [11]:
team_cursor = cursor.superstars.teams
aw = []
for documents in team_cursor.find({'created_at': {'$lt': end, '$gte': start}},{"user":1, "created_at":1}):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
df = pd.DataFrame(dic_flattened)
df.head()
teams = df[df["user"].isin(users["user_id"])]
teams = teams[["_id","user","created_at"]]
teams.columns = ["team_id", "user_id", "team_created_at"]
teams.head()

,team_id,user_id,team_created_at
0,5cc79098304ded1c8d247a1d,5cc79098304ded1c8d247a15,2019-04-30 00:02:32.279
1,5cc7986cc757b72b3a1c8689,5cc7986cc757b72b3a1c8681,2019-04-30 00:35:56.656
2,5cc798b6c1811e2b616907f7,5cc798b6c1811e2b616907ef,2019-04-30 00:37:10.234
3,5cc79982aaefa53d8f9ac78d,5cc79982aaefa53d8f9ac785,2019-04-30 00:40:34.013
4,5cc79b24c757b72b3a1c89f4,5cc79b24c757b72b3a1c89ec,2019-04-30 00:47:32.600


In [12]:
con_cursor = cursor.superstars.matches
aw = []
for documents in con_cursor.find({'created_at': {'$lt': end2, '$gte': start}},
                                 {"home_team":1, "winner_team":1,"status":1,"type":1,"start_time":1}):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
df = pd.DataFrame(dic_flattened)
df = df.rename(columns={'home_team_id':'team_id'})
matches = df[df["team_id"].isin(teams["team_id"])]
matches = matches.loc[:,["team_id","winner_team_id","status","type",'start_time']]
matches.head()

,team_id,winner_team_id,status,type,start_time
2,5cc79098304ded1c8d247a1d,5cc79098304ded1c8d247a1d,3,CAMPAIGN,2019-04-30 00:02:50.694
6,5cc79098304ded1c8d247a1d,5cc79098304ded1c8d247a1d,3,CAMPAIGN,2019-04-30 00:05:31.634
8,5cc79098304ded1c8d247a1d,5cc79098304ded1c8d247a1d,3,CAMPAIGN,2019-04-30 00:09:33.989
19,5cc79098304ded1c8d247a1d,5cc79098304ded1c8d247a1d,3,CAMPAIGN,2019-04-30 00:19:30.573
27,5cc79098304ded1c8d247a1d,5cc79098304ded1c8d247a1d,3,CAMPAIGN,2019-04-30 00:24:44.057


In [13]:
df_outer = pd.merge(teams, matches, on='team_id', how='inner')
print(len(df_outer))
df2 = pd.merge(df_outer,users,on='user_id',how = 'inner')
print(len(df2))
print(df2['type'].unique())

2068
2068
['CAMPAIGN' 'IPL' 'ENTRY_LEAGUE']


In [14]:
t = df2[df2['type'] != 'ENTRY_LEAGUE']
t['difference'] = t.start_time - t.createtime
t = t[t['difference']<='24:00:00']
t.sort_values('difference', ascending = False, inplace = True)
print(t.info())

/home/aurora/miniconda3/lib/python3.7/site-packages/ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  


<class 'pandas.core.frame.DataFrame'>
Int64Index: 1248 entries, 1679 to 1018
Data columns (total 11 columns):
team_id            1248 non-null object
user_id            1248 non-null object
team_created_at    1248 non-null datetime64[ns]
winner_team_id     946 non-null object
status             1248 non-null int64
type               1248 non-null object
start_time         1248 non-null datetime64[ns]
createtime         1248 non-null datetime64[ns]
device_id          1248 non-null object
last_request       1248 non-null datetime64[ns]
difference         1248 non-null timedelta64[ns]
dtypes: datetime64[ns](4), int64(1), object(5), timedelta64[ns](1)
memory usage: 117.0+ KB
None


In [15]:
qst = t.groupby('user_id')['start_time'].count()
q2 = qst.to_frame().reset_index()
q2.rename(columns={"start_time": "NOM"},inplace=True)
q2.sort_values('NOM',ascending =False,inplace = True)
t['tf'] = (t.status == 3)
mf = t.groupby('user_id')['tf'].sum()
q3 = mf.to_frame().reset_index()
q3.sort_values('tf',ascending =False,inplace = True)
a1 = pd.merge(q2,q3,on='user_id',how = 'outer')
a1['flag'] = a1.NOM - a1.tf
a1 = a1.sort_values('flag',ascending=False)
a1.flag.sum()
t['mw'] = (t.team_id == t.winner_team_id)
mw = t.groupby('user_id')['mw'].sum()
q4 = mw.to_frame().reset_index()
q4.sort_values('mw',ascending =False,inplace = True)
A = pd.merge(a1,q4,on='user_id',how = 'outer')
print(A.info())
print(len(A))

<class 'pandas.core.frame.DataFrame'>
Int64Index: 314 entries, 0 to 313
Data columns (total 5 columns):
user_id    314 non-null object
NOM        314 non-null int64
tf         314 non-null float64
flag       314 non-null float64
mw         314 non-null float64
dtypes: float64(3), int64(1), object(1)
memory usage: 14.7+ KB
None
314


In [16]:
B = pd.merge(A,users,on='user_id',how = 'outer')
print(len(B))

B['d1'] = ((B.last_request-B.createtime)>'24:00:00')
B.drop(['device_id','flag','createtime','last_request'], axis=1,inplace = True)
B.fillna(0)
total = B['d1'].sum()
print(total)
print((total/len(B))*100)

360
95
26.38888888888889


In [ ]:
B['temp'] = np.where(B['NOM']>=5, '5+', B['NOM'])
B.sort_values('NOM',inplace = True)
print(B.head())
print(len(B))
print(len(users))

In [18]:
C = B.groupby('temp')['user_id'].nunique()
C = C.to_frame().reset_index()
D = B.groupby('temp')['d1'].sum()
D = D.to_frame().reset_index()
E = pd.merge(C,D,on='temp',how = 'outer')
print(E.head())

  temp  user_id    d1
0  1.0      112  28.0
1  2.0       73  14.0
2  3.0       43  12.0
3  4.0       23   7.0
4   5+       63  30.0


In [19]:
E['Result'] = (E['d1']/E['user_id'])*100
print(E.head())


  temp  user_id    d1     Result
0  1.0      112  28.0  25.000000
1  2.0       73  14.0  19.178082
2  3.0       43  12.0  27.906977
3  4.0       23   7.0  30.434783
4   5+       63  30.0  47.619048
